# Testing over usecases


Populates the entire `results/` folder:
1. `results/sample_inputs/` — sample images used
2. `results/json_outputs/` — JSON extraction for 10+ documents
3. `results/eval_summary.csv` — evaluation metrics table
4. `results/qualitative/` — 5+ side-by-side input/output images

**Model:** Qwen3-VL-2B (~4 GB VRAM, ~12s/image)  

In [1]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless matplotlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.3 MB/s eta 0:00:00


In [3]:
# Cell 2: Clone repo + setup
import os, sys

REPO_DIR = "/content/VLM-PDF-Text_Extractor"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RahulReadd/VLM-PDF-Text_Extractor.git
os.chdir(REPO_DIR)
sys.path.insert(0, ".")

# Create results directories
for d in ["results/sample_inputs", "results/json_outputs", "results/qualitative"]:
    os.makedirs(d, exist_ok=True)

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CUDA: True
GPU: Tesla T4 | VRAM: 15.6 GB


In [4]:
# Cell 3: Load model
from app.extract import DocumentExtractor

MODEL_NAME = "qwen3-vl-2b"
extractor = DocumentExtractor(model_name=MODEL_NAME)
print(f"Model ready: {MODEL_NAME}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

DocumentExtractor ready: qwen3-vl-2b
Model ready: qwen3-vl-2b


---
## Part 1: CORD Receipt Extraction (12 documents)

In [5]:
# Cell 4: Load CORD dataset
import json
import numpy as np
from datasets import load_dataset
from app.evaluate import (
    evaluate_single, parse_cord_ground_truth,
    evaluate_signature_single, evaluate_signature_batch,
)

cord = load_dataset("naver-clova-ix/cord-v2", split="test")

CORD_N = 12
cord_indices = np.linspace(0, len(cord) - 1, CORD_N, dtype=int).tolist()
cord_images = [cord[i]["image"].convert("RGB") for i in cord_indices]
cord_gts = [cord[i]["ground_truth"] for i in cord_indices]

print(f"Loaded {len(cord_images)} CORD receipt images")

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00002-of-00004-688fe1305a55e5(…):   0%|          | 0.00/444M [00:00<?, ?B/s]

...

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Loaded 12 CORD receipt images


In [6]:
# Cell 5: Run extraction on CORD + evaluate
cord_results = []

for i, (img, gt_raw) in enumerate(zip(cord_images, cord_gts)):
    pred = extractor.extract(img)  # unified prompt
    gt_parsed = parse_cord_ground_truth(gt_raw)

    # For evaluation: unwrap receipt from unified output
    eval_json = pred["parsed_json"]
    if eval_json and "receipt" in eval_json and isinstance(eval_json["receipt"], dict):
        eval_json = eval_json["receipt"]

    scores = evaluate_single(eval_json, gt_parsed)

    result = {
        "image_idx": int(cord_indices[i]),
        "source": "cord",
        "model": MODEL_NAME,
        "prediction": pred["parsed_json"],
        "json_valid": pred["json_valid"],
        "scores": scores,
    }
    cord_results.append(result)

    # Save JSON output
    out_path = f"results/json_outputs/cord_{i:03d}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    # Save sample input image
    img.save(f"results/sample_inputs/cord_{i:03d}.png")

    print(f"  [{i+1}/{CORD_N}] JSON={pred['json_valid']} | Field F1={scores['field_f1']:.3f} | Menu F1={scores['menu_f1']:.3f}")

# Aggregate CORD metrics
cord_n = len(cord_results)
cord_json_rate = sum(r["json_valid"] for r in cord_results) / cord_n
cord_field_em = sum(r["scores"]["field_em"] for r in cord_results) / cord_n
cord_field_f1 = sum(r["scores"]["field_f1"] for r in cord_results) / cord_n
cord_menu_f1 = sum(r["scores"]["menu_f1"] for r in cord_results) / cord_n

print(f"\nCORD Results ({cord_n} images):")
print(f"  JSON valid:  {cord_json_rate:.0%}")
print(f"  Field EM:    {cord_field_em:.3f}")
print(f"  Field F1:    {cord_field_f1:.3f}")
print(f"  Menu F1:     {cord_menu_f1:.3f}")

  [1/12] JSON=True | Field F1=0.667 | Menu F1=0.000
  [2/12] JSON=True | Field F1=0.000 | Menu F1=0.000
  [3/12] JSON=True | Field F1=1.000 | Menu F1=0.000
  [4/12] JSON=True | Field F1=0.333 | Menu F1=1.000
  [5/12] JSON=True | Field F1=1.000 | Menu F1=1.000
  [6/12] JSON=True | Field F1=0.000 | Menu F1=1.000
  [7/12] JSON=True | Field F1=1.000 | Menu F1=1.000
  [8/12] JSON=True | Field F1=1.000 | Menu F1=0.500
  [9/12] JSON=True | Field F1=0.333 | Menu F1=1.000
  [10/12] JSON=True | Field F1=1.000 | Menu F1=1.000
  [11/12] JSON=True | Field F1=0.333 | Menu F1=1.000
  [12/12] JSON=True | Field F1=0.000 | Menu F1=0.308

CORD Results (12 images):
  JSON valid:  100%
  Field EM:    0.556
  Field F1:    0.556
  Menu F1:     0.651


---
## Part 2: Signature Detection Evaluation (20 documents)

In [7]:
# Cell 6: Load signature dataset (positive) + CORD as negative
SIG_N = 10  # per class → 20 total

# Positive: documents WITH signatures
sig_ds = load_dataset("Mels22/SigDetectVerifyFlow", split="test")
sig_pos_indices = np.linspace(0, len(sig_ds) - 1, SIG_N, dtype=int).tolist()
sig_pos_images = [sig_ds[i]["document"].convert("RGB") for i in sig_pos_indices]

# Negative: CORD receipts (no signatures) — use different indices than Part 1
sig_neg_indices = np.linspace(0, len(cord) - 1, SIG_N, dtype=int).tolist()
sig_neg_images = [cord[i]["image"].convert("RGB") for i in sig_neg_indices]

sig_all_images = sig_pos_images + sig_neg_images
sig_all_labels = [True] * SIG_N + [False] * SIG_N

print(f"Loaded {len(sig_all_images)} images ({SIG_N} with signatures, {SIG_N} without)")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/407M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/413M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

...

data/test-00000-of-00002.parquet:   0%|          | 0.00/337M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23206 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6195 [00:00<?, ? examples/s]

Loaded 20 images (10 with signatures, 10 without)


In [8]:
# Cell 7: Run signature detection + evaluate
sig_predictions = []
sig_results_raw = []

for i, (img, gt_label) in enumerate(zip(sig_all_images, sig_all_labels)):
    pred = extractor.extract(img)  # unified prompt
    sig_predictions.append(pred["parsed_json"])

    source = "sigdetect" if i < SIG_N else "cord"
    result = {
        "image_idx": i,
        "source": source,
        "model": MODEL_NAME,
        "gt_signature_present": gt_label,
        "prediction": pred["parsed_json"],
        "json_valid": pred["json_valid"],
    }
    sig_results_raw.append(result)

    # Save JSON
    out_path = f"results/json_outputs/sig_{i:03d}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    # Save sample input
    img.save(f"results/sample_inputs/sig_{i:03d}.png")

    tag = "+" if gt_label else "-"
    print(f"  [{tag}] {i+1}/{len(sig_all_images)} ({source}): JSON={pred['json_valid']}")

# Compute metrics
sig_metrics = evaluate_signature_batch(sig_predictions, sig_all_labels)

print(f"\nSignature Detection Results ({sig_metrics['n']} images):")
print(f"  Accuracy:  {sig_metrics['accuracy']:.3f}")
print(f"  Precision: {sig_metrics['precision']:.3f}")
print(f"  Recall:    {sig_metrics['recall']:.3f}")
print(f"  F1:        {sig_metrics['f1']:.3f}")
print(f"  TP={sig_metrics['tp']}  FP={sig_metrics['fp']}  FN={sig_metrics['fn']}  TN={sig_metrics['tn']}")

  [+] 1/20 (sigdetect): JSON=True
  [+] 2/20 (sigdetect): JSON=True
  [+] 3/20 (sigdetect): JSON=True
  [+] 4/20 (sigdetect): JSON=True
  [+] 5/20 (sigdetect): JSON=True
  [+] 6/20 (sigdetect): JSON=True
  [+] 7/20 (sigdetect): JSON=True
  [+] 8/20 (sigdetect): JSON=False
  [+] 9/20 (sigdetect): JSON=True
  [+] 10/20 (sigdetect): JSON=True
  [-] 11/20 (cord): JSON=True
  [-] 12/20 (cord): JSON=True
  [-] 13/20 (cord): JSON=True
  [-] 14/20 (cord): JSON=True
  [-] 15/20 (cord): JSON=True
  [-] 16/20 (cord): JSON=True
  [-] 17/20 (cord): JSON=True
  [-] 18/20 (cord): JSON=True
  [-] 19/20 (cord): JSON=True
  [-] 20/20 (cord): JSON=False

Signature Detection Results (20 images):
  Accuracy:  0.950
  Precision: 1.000
  Recall:    0.900
  F1:        0.947
  TP=9  FP=0  FN=1  TN=10


---
## Part 3: Evaluation Metrics Summary (CSV)

In [9]:
# Cell 8: Save evaluation summary CSV
import pandas as pd

summary = pd.DataFrame([
    {
        "Task": "Receipt Extraction (CORD)",
        "Dataset": "naver-clova-ix/cord-v2",
        "N": cord_n,
        "Model": MODEL_NAME,
        "JSON_Valid_Rate": round(cord_json_rate, 3),
        "Field_EM": round(cord_field_em, 3),
        "Field_F1": round(cord_field_f1, 3),
        "Menu_F1": round(cord_menu_f1, 3),
        "Accuracy": "",
        "Precision": "",
        "Recall": "",
        "Sig_F1": "",
    },
    {
        "Task": "Signature Detection",
        "Dataset": "SigDetectVerifyFlow + CORD",
        "N": sig_metrics["n"],
        "Model": MODEL_NAME,
        "JSON_Valid_Rate": "",
        "Field_EM": "",
        "Field_F1": "",
        "Menu_F1": "",
        "Accuracy": sig_metrics["accuracy"],
        "Precision": sig_metrics["precision"],
        "Recall": sig_metrics["recall"],
        "Sig_F1": sig_metrics["f1"],
    },
])

csv_path = "results/eval_summary.csv"
summary.to_csv(csv_path, index=False)
print(f"Saved evaluation summary to: {csv_path}")
print()
print(summary.to_string(index=False))

Saved evaluation summary to: results/eval_summary.csv

                     Task                    Dataset  N       Model JSON_Valid_Rate Field_EM Field_F1 Menu_F1 Accuracy Precision Recall Sig_F1
Receipt Extraction (CORD)     naver-clova-ix/cord-v2 12 qwen3-vl-2b             1.0    0.556    0.556   0.651                                 
      Signature Detection SigDetectVerifyFlow + CORD 20 qwen3-vl-2b                                               0.95       1.0    0.9  0.947


---
## Part 4: Qualitative Side-by-Side Examples (5+)

In [10]:
# Cell 9: Generate side-by-side qualitative images
import matplotlib.pyplot as plt
import textwrap


def make_qualitative(image, pred_json, title, ground_truth_text, save_path):
    """Create a side-by-side PNG: input image | extracted JSON | ground truth."""
    fig, axes = plt.subplots(1, 3, figsize=(24, 8),
                             gridspec_kw={"width_ratios": [1, 1, 1]})

    # Left: input image
    axes[0].imshow(image)
    axes[0].set_title("Input Image", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    # Middle: model prediction
    pred_text = json.dumps(pred_json, indent=2, ensure_ascii=False) if pred_json else "(no valid JSON)"
    pred_wrapped = textwrap.fill(pred_text, width=60)
    if len(pred_wrapped) > 1500:
        pred_wrapped = pred_wrapped[:1500] + "\n... [truncated]"
    axes[1].text(0.02, 0.98, pred_wrapped, transform=axes[1].transAxes,
                 fontsize=7, verticalalignment="top", fontfamily="monospace",
                 bbox=dict(boxstyle="round", facecolor="#e8f5e9", alpha=0.8))
    axes[1].set_title("Model Prediction", fontsize=14, fontweight="bold")
    axes[1].axis("off")

    # Right: ground truth
    gt_wrapped = textwrap.fill(ground_truth_text, width=60)
    if len(gt_wrapped) > 1500:
        gt_wrapped = gt_wrapped[:1500] + "\n... [truncated]"
    axes[2].text(0.02, 0.98, gt_wrapped, transform=axes[2].transAxes,
                 fontsize=7, verticalalignment="top", fontfamily="monospace",
                 bbox=dict(boxstyle="round", facecolor="#e3f2fd", alpha=0.8))
    axes[2].set_title("Ground Truth", fontsize=14, fontweight="bold")
    axes[2].axis("off")

    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {save_path}")


print("Generating qualitative examples...\n")

Generating qualitative examples...



In [11]:
# Cell 10: Generate 3 CORD receipt examples
print("--- CORD Receipt Examples ---")
for i in range(3):
    img = cord_images[i]
    pred = cord_results[i]["prediction"]
    gt_parsed = parse_cord_ground_truth(cord_gts[i])
    gt_text = json.dumps(gt_parsed, indent=2, ensure_ascii=False)
    scores = cord_results[i]["scores"]

    title = (f"CORD Receipt #{i+1} — "
             f"Field F1: {scores['field_f1']:.3f} | Menu F1: {scores['menu_f1']:.3f}")

    make_qualitative(
        image=img,
        pred_json=pred,
        title=title,
        ground_truth_text=gt_text,
        save_path=f"results/qualitative/cord_example_{i+1}.png",
    )

--- CORD Receipt Examples ---
  Saved: results/qualitative/cord_example_1.png
  Saved: results/qualitative/cord_example_2.png
  Saved: results/qualitative/cord_example_3.png


In [12]:
# Cell 11: Generate 2 signature detection examples (1 positive, 1 negative)
from app.evaluate import parse_signature_prediction

print("--- Signature Detection Examples ---")

# Example 1: Document WITH signature (positive)
pos_idx = 0
pos_img = sig_all_images[pos_idx]
pos_pred = sig_predictions[pos_idx]
pos_detected = parse_signature_prediction(pos_pred)
pos_gt = "Ground Truth: Signature PRESENT"
pos_title = f"Signature Doc (Positive) — Detected: {pos_detected} | GT: True"

make_qualitative(
    image=pos_img,
    pred_json=pos_pred,
    title=pos_title,
    ground_truth_text=pos_gt,
    save_path="results/qualitative/sig_positive_example.png",
)

# Example 2: Receipt WITHOUT signature (negative)
neg_idx = SIG_N  # first negative sample
neg_img = sig_all_images[neg_idx]
neg_pred = sig_predictions[neg_idx]
neg_detected = parse_signature_prediction(neg_pred)
neg_gt = "Ground Truth: Signature NOT PRESENT"
neg_title = f"Receipt (Negative) — Detected: {neg_detected} | GT: False"

make_qualitative(
    image=neg_img,
    pred_json=neg_pred,
    title=neg_title,
    ground_truth_text=neg_gt,
    save_path="results/qualitative/sig_negative_example.png",
)

--- Signature Detection Examples ---
  Saved: results/qualitative/sig_positive_example.png
  Saved: results/qualitative/sig_negative_example.png


In [13]:
# Cell 12: Generate 1 more CORD example with full unified output visible
print("--- Full Unified Extraction Example ---")
i = 5  # pick a different CORD image
img = cord_images[i]
pred = cord_results[i]["prediction"]
gt_parsed = parse_cord_ground_truth(cord_gts[i])
gt_text = json.dumps(gt_parsed, indent=2, ensure_ascii=False)
scores = cord_results[i]["scores"]

title = (f"Unified Extraction Example — All 4 aspects in one pass\n"
         f"Field F1: {scores['field_f1']:.3f} | Menu F1: {scores['menu_f1']:.3f}")

make_qualitative(
    image=img,
    pred_json=pred,
    title=title,
    ground_truth_text=gt_text,
    save_path="results/qualitative/unified_example.png",
)

--- Full Unified Extraction Example ---
  Saved: results/qualitative/unified_example.png


---
## Part 5: Final Summary

In [14]:
# Cell 13: Print final summary of everything generated
from pathlib import Path

print("=" * 60)
print("  RESULTS FOLDER — SUBMISSION READY")
print("=" * 60)

json_count = len(list(Path("results/json_outputs").glob("*.json")))
sample_count = len(list(Path("results/sample_inputs").glob("*.png")))
qual_count = len(list(Path("results/qualitative").glob("*.png")))
csv_exists = Path("results/eval_summary.csv").exists()

print(f"\n  results/json_outputs/    : {json_count} JSON files (required: 10+) {'OK' if json_count >= 10 else 'NEED MORE'}")
print(f"  results/sample_inputs/   : {sample_count} sample images")
print(f"  results/qualitative/     : {qual_count} side-by-side PNGs (required: 5+) {'OK' if qual_count >= 5 else 'NEED MORE'}")
print(f"  results/eval_summary.csv : {'EXISTS' if csv_exists else 'MISSING'}")

print(f"\n  Model: {MODEL_NAME}")
print(f"  Total documents processed: {json_count}")

print(f"\n{'─'*60}")
print(f"  Receipt Extraction (CORD, n={cord_n}):")
print(f"    JSON valid: {cord_json_rate:.0%}")
print(f"    Field EM:   {cord_field_em:.3f}")
print(f"    Field F1:   {cord_field_f1:.3f}")
print(f"    Menu F1:    {cord_menu_f1:.3f}")

print(f"\n  Signature Detection (n={sig_metrics['n']}):")
print(f"    Accuracy:   {sig_metrics['accuracy']:.3f}")
print(f"    Precision:  {sig_metrics['precision']:.3f}")
print(f"    Recall:     {sig_metrics['recall']:.3f}")
print(f"    F1:         {sig_metrics['f1']:.3f}")
print(f"{'─'*60}")

# List all generated files
print(f"\n  All generated files:")
for root, dirs, files in os.walk("results"):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"    {path} ({size:,} bytes)")

  RESULTS FOLDER — SUBMISSION READY

  results/json_outputs/    : 32 JSON files (required: 10+) OK
  results/sample_inputs/   : 32 sample images
  results/qualitative/     : 6 side-by-side PNGs (required: 5+) OK
  results/eval_summary.csv : EXISTS

  Model: qwen3-vl-2b
  Total documents processed: 32

────────────────────────────────────────────────────────────
  Receipt Extraction (CORD, n=12):
    JSON valid: 100%
    Field EM:   0.556
    Field F1:   0.556
    Menu F1:    0.651

  Signature Detection (n=20):
    Accuracy:   0.950
    Precision:  1.000
    Recall:     0.900
    F1:         0.947
────────────────────────────────────────────────────────────

  All generated files:
    results/README.md (652 bytes)
    results/eval_summary.csv (271 bytes)
    results/sample_inputs/cord_000.png (293,212 bytes)
    results/sample_inputs/cord_001.png (711,154 bytes)
    results/sample_inputs/cord_002.png (731,229 bytes)
    results/sample_inputs/cord_003.png (976,002 bytes)
    results/sam

In [15]:
import shutil
from google.colab import files

folder_to_download = '/content/VLM-PDF-Text_Extractor/results'
output_filename = 'results_archive'

# Create a zip archive of the folder
shutil.make_archive(output_filename, 'zip', folder_to_download)

# Download the zip file
files.download(output_filename + '.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>